# nb_mig_silver_20261117163000_SeedRegSub_TransformMappings_Unified_C

Create the metadata entried for the various silver unified tables for Registration Submissions. 

Note, since multiple developers are working on this simultaneously, the metadata insertion has been divided across multiple migration notebooks

In [ ]:
%run "/05 - shared-libraries/nb_shared_common"

In [ ]:
%run "/05 - shared-libraries/nb_shared_metadata"

In [ ]:
print("read_transform_entity_mapping" in dir())
print("get_next_record_id" in dir())
print("get_max_transform_order" in dir())
print("verify_unique_records" in dir())

In [ ]:
get_max_transform_order(
    "metadata2_db",
    "silver_master_unify"
)

In [ ]:
UNIFIED_MASTER_CURRENT_MAX_ORDER = get_max_transform_order(
    "metadata2_db",
    STEP_SILVER_MASTER_UNIFY
)

UNIFIED_TX_CURRENT_MAX_ORDER = get_max_transform_order(
    "metadata2_db",
    STEP_SILVER_TX_UNIFY
)
print (UNIFIED_MASTER_CURRENT_MAX_ORDER)
print (UNIFIED_TX_CURRENT_MAX_ORDER)

In [ ]:
%run "/05 - shared-libraries/nb_shared_migrations"

In [ ]:
storage_account_name = "deintdlsaueaasqebla"
dataverse_container = "dataverse-asqaarms01de-unqde396ad0f22d448c94c779e588420"

In [ ]:
#imports
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime, timezone
from delta.tables import DeltaTable
from mdw_shared.migrations import ensure_history_table, is_applied, record_migration

# Get spark session and delta lake paths
spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name, dataverse_container)
METADATA_BASE_PATH = data_lake_paths["metadata"]

VERSION_ID = "v1"

MIGRATION_ID = "20261117163000_SeedRegSub_TransformMappings_Unified_C"
NOTEBOOK_NAME = "nb_mig_silver_20261117163000_SeedRegSub_TransformMappings_Unified_C"

ensure_history_table(spark, METADATA_BASE_PATH)


In [ ]:
if is_applied(spark, METADATA_BASE_PATH, MIGRATION_ID):
    print(f"Already applied: {MIGRATION_ID}")
else:
    now = datetime.now(timezone.utc)

    is_active = True
    created_utc = now
    updated_utc = now

    # =================================================================
    # PART 1: Transform Mappings
    # =================================================================

    NOTEBOOK_TO_EXECUTE_NONE = None
    JSON_CONFIG_NONE = None
    SOURCE_ID_NONE = None
    SOURCE_ENTITY_NONE = None

    # The order is no longer typed in by hand. Rows below carry NO order value; assign_orders()
    # numbers them sequentially per step, continuing from the highest order already in the metadata
    # table when this migration runs. That keeps the numbering unique and gap-free no matter how many
    # migration notebooks are adding rows, and nobody needs to know what the other notebooks used.
    # NOTE: this only guarantees unique, sequential numbers. Within each list, put parents before
    # children (the order here is the run order).

    def assign_orders(rows, start_by_step):
        counters = dict(start_by_step)
        out = []
        for binding, step, source_id, source_entity, target_entity, notebook, json_file in rows:
            counters[step] += 1
            out.append((binding, step, source_id, counters[step], source_entity, target_entity, notebook, json_file))
        return out

    # Read the current max at execution time, not at notebook-write time
    start_by_step = {
        STEP_SILVER_MASTER_UNIFY: get_max_transform_order("metadata2_db", STEP_SILVER_MASTER_UNIFY) or 0,
        STEP_SILVER_TX_UNIFY:     get_max_transform_order("metadata2_db", STEP_SILVER_TX_UNIFY) or 0,
    }

    # transform_entity_mappings contains the metadata entries to map ARMS to Clease
    # and cleanse to unify
    # Parameter 1 : Binding Id
    # Parameter 2 : Step Name
    # Parameter 3 : Source Id (only relevant for ARMS cleanse, for unifcation this is set to SOURCE_ID_NONE)
    # Parameter 4 : Source entity name
    # Parameter 5 : Target entity name
    # Parameter 6 : Any custom notebook to run for the transformation only if relevant
    # Parameter 7 : name of JSON mapping file to map fields and run relevant validations at a field level
    # (Order is added by assign_orders)

    master_rows = [
         # Unify Master
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, "asqa_governingpersonrole", "governing_person_role", NOTEBOOK_TO_EXECUTE_NONE, "governing_person_role.json"),
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, "asqa_rtotypeofsignificantevent", "rto_type_of_significant_event", NOTEBOOK_TO_EXECUTE_NONE, "rto_type_of_significant_event.json"),
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, "asqa_studentmanagementprovider", "student_management_provider", NOTEBOOK_TO_EXECUTE_NONE, "student_management_provider.json"),
    ]

    tx_rows = [
         # Unify Transaction
         # asqa_registrationsubmission is the parent of the reg_sub_* tables below, so it goes first
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_asqainvoice", "invoice", NOTEBOOK_TO_EXECUTE_NONE, "invoice.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_registrationsubmission", "registration_submission", NOTEBOOK_TO_EXECUTE_NONE, "registration_submission.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rfi", "reg_sub_rfi", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rfi.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtodelegation", "reg_sub_rto_delegation", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_delegation.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtoevidenceofdelivery", "reg_sub_rto_evidence_of_delivery", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_evidence_of_delivery.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtofinancialstatus1", "reg_sub_rto_financial_status", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_financial_status.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtolegaldetails", "reg_sub_rto_legal_detail", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_legal_detail.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtoownership", "reg_sub_rto_ownership", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_ownership.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_thirdpartyarrangements", "reg_sub_third_party_arrangement", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_third_party_arrangement.json"),
    ]

    transform_entity_mappings = assign_orders(master_rows, start_by_step) + assign_orders(tx_rows, start_by_step)

    for m in transform_entity_mappings:
        print(m[1], m[3], m[4], "->", m[5])

    insert_transform_entity_mappings(METADATA_BASE_PATH, transform_entity_mappings, VERSION_ID, spark)

    # TODO: remove before PR
    #record_migration(spark, METADATA_BASE_PATH, MIGRATION_ID, BINDING_STAGE_SILVER, NOTEBOOK_NAME, "succeeded")
    print(f"Applied: {MIGRATION_ID}")


In [ ]:
UNIFIED_MASTER_CURRENT_MAX_ORDER = get_max_transform_order(
    "metadata2_db",
    STEP_SILVER_MASTER_UNIFY
)

UNIFIED_TX_CURRENT_MAX_ORDER = get_max_transform_order(
    "metadata2_db",
    STEP_SILVER_TX_UNIFY
)
print (UNIFIED_MASTER_CURRENT_MAX_ORDER)
print (UNIFIED_TX_CURRENT_MAX_ORDER)

In [ ]:
%%sql
SELECT *
FROM metadata2_db.transform_businessready_entity_mapping
WHERE step_name in ('silver_master_unify')
and source_entity_name like '%student%'
order by step_name, order DESC;

In [ ]:
%%sql
SELECT *
FROM metadata2_db.transform_businessready_entity_mapping
WHERE step_name in ( 'silver_transactions_unify')
and source_entity_name like '%rfi%'
order by step_name, order DESC;

In [ ]:
spark.sql("""
DESCRIBE TABLE metadata2_db.transform_businessready_entity_mapping
""").filter("col_name = 'order'").show(truncate=False)